# 01 PyTorch 基础：Tensor、autograd 与线性回归

这份 notebook 是深度学习阶段一的起点。你的已有背景是：已经学过 Python、NumPy/Pandas 和传统机器学习，所以这里不从 Python 语法重新开始，而是直接进入 PyTorch 的核心用法。

推荐运行环境：`D:\\PythonWorkSpace\\anaconda\\envs\\pytorch\\python.exe`。当前检测到这个环境可以正常导入 PyTorch。显卡可能需要更新版 PyTorch 才能完整使用 CUDA，所以本节代码会优先保证 CPU 可运行。

本节目标：

- 会创建、索引、切片、变形 Tensor。
- 理解广播机制和 Tensor 形状。
- 会在 CPU / CUDA 设备之间选择。
- 理解 `requires_grad`、`backward()` 和梯度。
- 跑通一个最小 PyTorch 训练循环。

## 0. 环境检查

如果下面单元格导入 `torch` 失败，先不要着急。当前机器的 `dl-study` 环境中可能存在 Windows 应用程序控制策略拦截 `torch.dll` 的情况。先记录错误信息，再处理环境。

In [ ]:
import sys

print(sys.executable)
print(sys.version)

try:
    import torch
    print("torch version:", torch.__version__)
    print("cuda available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("cuda device:", torch.cuda.get_device_name(0))
except Exception as e:
    print(type(e).__name__)
    print(e)

## 1. Tensor 创建

Tensor 可以理解为 PyTorch 里的多维数组，是模型输入、参数、输出、损失计算的基础数据结构。

In [ ]:
import torch

a = torch.tensor([1, 2, 3])
b = torch.zeros(2, 3)
c = torch.ones(2, 3)
d = torch.randn(2, 3)
e = torch.arange(0, 10, 2)

print("a =", a)
print("b.shape =", b.shape)
print("c.dtype =", c.dtype)
print("d =", d)
print("e =", e)

常见形状约定：

- 表格数据：`[batch_size, num_features]`
- 灰度图像：`[batch_size, 1, height, width]`
- 彩色图像：`[batch_size, 3, height, width]`
- 文本序列：常见为 `[batch_size, seq_len]` 或 `[seq_len, batch_size]`，取决于模型接口

## 2. 索引、切片与变形

深度学习调试时，第一优先级通常是看清楚张量形状。很多模型错误都来自维度不匹配。

In [ ]:
x = torch.arange(12).reshape(3, 4)

print(x)
print("第一行:", x[0])
print("第一列:", x[:, 0])
print("前两行、后两列:\n", x[:2, 2:])

y = x.reshape(2, 6)
z = x.flatten()

print("y.shape =", y.shape)
print("z.shape =", z.shape)

## 3. 广播机制

广播允许不同形状的 Tensor 在满足规则时自动扩展后计算。它很方便，但也容易让错误悄悄发生，所以每一步都要看 shape。

In [ ]:
x = torch.ones(3, 4)
bias = torch.tensor([10, 20, 30, 40])

print("x.shape =", x.shape)
print("bias.shape =", bias.shape)
print(x + bias)

## 4. Tensor 与 NumPy 转换

在 CPU 上，`tensor.numpy()` 和 `torch.from_numpy()` 可能共享内存。修改其中一个，另一个也可能变化。

In [ ]:
import numpy as np

np_array = np.array([1.0, 2.0, 3.0], dtype=np.float32)
tensor_from_np = torch.from_numpy(np_array)

tensor_to_np = tensor_from_np.numpy()

np_array[0] = 100.0

print("np_array:", np_array)
print("tensor_from_np:", tensor_from_np)
print("tensor_to_np:", tensor_to_np)

## 5. 设备选择：CPU / CUDA

模型和数据必须在同一个设备上。初学阶段先写出通用设备选择代码，后面训练 CNN/RNN 时会反复用到。

In [ ]:
def get_default_device():
    if not torch.cuda.is_available():
        return torch.device("cpu")

    try:
        test_tensor = torch.ones(1, device="cuda")
        _ = test_tensor + 1
        torch.cuda.synchronize()
        return torch.device("cuda")
    except Exception as e:
        print("CUDA detected, but current PyTorch cannot use it safely.")
        print(type(e).__name__, e)
        return torch.device("cpu")


device = get_default_device()
print("selected device:", device)

x = torch.randn(2, 3).to(device)
print(x.device)

## 6. autograd 自动求导

PyTorch 会记录带有 `requires_grad=True` 的 Tensor 参与过的计算，然后通过 `backward()` 自动计算梯度。

In [ ]:
x = torch.tensor(2.0, requires_grad=True)
y = x ** 2 + 3 * x + 1

y.backward()

print("y =", y.item())
print("dy/dx =", x.grad.item())

手算验证：

$$y = x^2 + 3x + 1$$

$$\frac{dy}{dx} = 2x + 3$$

当 `x = 2` 时，梯度应该是 `7`。

## 7. 用 PyTorch 重写线性回归

任务：生成一批近似满足 `y = 3x + 2` 的数据，让模型自己学习出权重和偏置。这个例子对应传统机器学习里的线性回归，但训练方式已经是深度学习通用套路：

1. 准备数据。
2. 定义模型。
3. 定义损失函数。
4. 定义优化器。
5. 循环执行 forward、loss、backward、step。

In [ ]:
import torch
from torch import nn

torch.manual_seed(42)

num_samples = 200
x = torch.linspace(-3, 3, num_samples).reshape(-1, 1)
noise = 0.3 * torch.randn_like(x)
y = 3 * x + 2 + noise

print("x.shape =", x.shape)
print("y.shape =", y.shape)
print("first 5 rows:")
print(torch.cat([x[:5], y[:5]], dim=1))

In [ ]:
model = nn.Linear(in_features=1, out_features=1)
loss_fn = nn.MSELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.05)

print(model)
print("initial weight:", model.weight.item())
print("initial bias:", model.bias.item())

In [ ]:
loss_history = []

for epoch in range(100):
    predictions = model(x)              # forward: 计算预测值
    loss = loss_fn(predictions, y)       # 计算损失

    optimizer.zero_grad()                # 清空上一轮梯度
    loss.backward()                      # backward: 自动计算梯度
    optimizer.step()                     # 根据梯度更新参数

    loss_history.append(loss.item())

    if (epoch + 1) % 20 == 0:
        print(f"epoch {epoch + 1:03d} | loss = {loss.item():.4f}")

print("learned weight:", model.weight.item())
print("learned bias:", model.bias.item())

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(6, 4))
plt.plot(loss_history)
plt.xlabel("epoch")
plt.ylabel("MSE loss")
plt.title("Training loss")
plt.grid(True)
plt.show()

In [ ]:
with torch.no_grad():
    y_pred = model(x)

plt.figure(figsize=(6, 4))
plt.scatter(x.numpy(), y.numpy(), s=12, label="data")
plt.plot(x.numpy(), y_pred.numpy(), color="red", label="model")
plt.xlabel("x")
plt.ylabel("y")
plt.title("Linear regression with PyTorch")
plt.legend()
plt.grid(True)
plt.show()

## 8. 训练循环关键词

- `forward()`：模型根据当前参数从输入计算输出。
- `loss_fn(predictions, y)`：衡量预测值和真实值之间的差距。
- `optimizer.zero_grad()`：清空历史梯度。PyTorch 默认会累加梯度。
- `loss.backward()`：根据损失自动计算每个参数的梯度。
- `optimizer.step()`：优化器根据梯度更新参数。

## 9. 本节自检

完成后用自己的话回答：

1. Tensor 和 NumPy array 的核心区别是什么？
2. 为什么训练前要关注输入输出 shape？
3. `loss.backward()` 做了什么？
4. 为什么每轮训练都要 `optimizer.zero_grad()`？
5. 线性回归实验里，模型学到的 weight 和 bias 应该接近多少？

## 10. 下一步

下一份 notebook 建议继续做：

- `Dataset` 与 `DataLoader`
- 训练集、验证集、测试集划分
- 二分类 MLP
- accuracy、loss 曲线记录